# Applied Machine Learning (CSAI2017P) — Lab Assignment 7
## Credit Risk Assessment

Experiments 9 · CO2, CO4 · **10 marks**

| | |
|---|---|
| Name | *Shardul Aggarwal* |
| SAP ID | *590022064* |
| Batch | *5* |
| Due | announced in the lab |
| Lab quiz | LQ7, after submission |

**Datasets:** A3 (Statlog German Credit)

---

### Before you start
- Attempt **2 of 3** in Section A, **1 of 2** in Section B. Section C is compulsory.
- Fit every transformer inside a `Pipeline`. Never fit on the test set.
- Set `random_state` wherever the question asks for reproducibility.
- Every question wants a short written observation, not just a number.
- Restart the kernel and run top-to-bottom before you submit.


In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sklearn

RANDOM_STATE = 0
np.random.seed(RANDOM_STATE)

for m in (np, pd, sklearn):
    print(f"{m.__name__:12s} {m.__version__}")


numpy        2.1.3
pandas       2.2.3
sklearn      1.6.1


## Load the data

Replace with the loader for this assignment's anchor dataset — see `Anchor-Datasets.pdf` for the snippets.

In [2]:
import os
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score, recall_score, confusion_matrix, roc_curve, roc_auc_score

# Dataset A3: Statlog German Credit (OpenML "credit-g", version 1; UCI Statlog)
LOCAL = "data/credit-g.csv"
if os.path.exists(LOCAL):
    df = pd.read_csv(LOCAL)
else:
    df = fetch_openml("credit-g", version=1, as_frame=True, parser="auto").frame
    os.makedirs("data", exist_ok=True)
    df.to_csv(LOCAL, index=False)

# Positive class = BAD credit (the costly one to miss). good -> 0, bad -> 1
y = (df["class"] == "bad").astype(int)
X = df.drop(columns="class")

num_cols = X.select_dtypes(include="number").columns.tolist()
cat_cols = [c for c in X.columns if c not in num_cols]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE)

def make_pipe(**lr_kwargs):
    """One-hot the categoricals, scale the numerics, then logistic regression.
    Everything is fitted inside the pipeline, so nothing leaks from the test set."""
    pre = ColumnTransformer([("num", StandardScaler(), num_cols),
                             ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols)])
    return Pipeline([("pre", pre),
                     ("clf", LogisticRegression(max_iter=2000, random_state=RANDOM_STATE, **lr_kwargs))])

# Cost matrix from the dataset documentation:
#   lending to a bad customer (predict good, truth bad)  = 5
#   refusing a good customer  (predict bad, truth good)  = 1
COST_FN, COST_FP = 5, 1
def total_cost(y_true, y_pred):
    y_true, y_pred = np.asarray(y_true), np.asarray(y_pred)
    fn = int(((y_true == 1) & (y_pred == 0)).sum())
    fp = int(((y_true == 0) & (y_pred == 1)).sum())
    return COST_FN * fn + COST_FP * fp

print(df.shape, "| train:", len(X_train), "| test:", len(X_test))
print("numeric:", len(num_cols), "| categorical:", len(cat_cols))


(1000, 21) | train: 800 | test: 200
numeric: 7 | categorical: 13


---

# Section A — attempt any 2 of 3 (2 marks each)

*attempt 2 of 3*

## A1. Load and baseline  &nbsp;&nbsp;`[2 marks]`

*Dataset: A3 (credit-g). Target: `class`.*

a. Fetch the dataset, report its shape and the proportion of bad-credit cases.
b. Fit a logistic regression in a pipeline with one-hot encoding and scaling; report accuracy.
c. State the majority-class accuracy and compare in two lines.


In [3]:
# a. shape and proportion of bad-credit cases
print("shape:", df.shape)
print(f"bad credit : {y.sum()} of {len(y)}  ({y.mean():.1%})")
print(f"good credit: {(y == 0).sum()} of {len(y)}  ({1 - y.mean():.1%})")

# b. logistic regression baseline (one-hot + scaling in a pipeline)
base = make_pipe().fit(X_train, y_train)
pred_base = base.predict(X_test)
acc_base = accuracy_score(y_test, pred_base)
print(f"\nLogistic regression test accuracy: {acc_base:.4f}")

# c. majority-class baseline
dummy = DummyClassifier(strategy="most_frequent").fit(X_train, y_train)
acc_dummy = accuracy_score(y_test, dummy.predict(X_test))
print(f"Majority-class (always 'good') accuracy: {acc_dummy:.4f}")
print(f"Gain over majority baseline: {acc_base - acc_dummy:+.4f}")


shape: (1000, 21)
bad credit : 300 of 1000  (30.0%)
good credit: 700 of 1000  (70.0%)

Logistic regression test accuracy: 0.7450
Majority-class (always 'good') accuracy: 0.7000
Gain over majority baseline: +0.0450


**Observation (A1):**

> **Numbers to quote after running:** shape, % bad, logistic-regression accuracy, majority-class accuracy.
> The classes are imbalanced (about 30% bad), so always predicting "good" already scores about 70%. The logistic regression is only useful to the extent that it beats that baseline, so report the gain, not the raw accuracy.
> Even a few points above the baseline matters here, because the gain comes from catching some of the costly bad-credit cases.


## A2. The cost matrix  &nbsp;&nbsp;`[2 marks]`

*Dataset: A3.*

a. Print the confusion matrix for your A1 model.
b. The dataset's own documentation says lending to a bad customer costs five times as much as refusing a good one. Compute your model's total cost under that matrix.
c. State in two lines whether accuracy was the right metric.


In [4]:
# a. confusion matrix of the A1 model (rows = true, cols = predicted)
cm = confusion_matrix(y_test, pred_base)
print(pd.DataFrame(cm, index=["true good", "true bad"], columns=["pred good", "pred bad"]))

# b. total cost under the 5:1 matrix
fn = cm[1, 0]   # bad customer predicted good  -> cost 5 each
fp = cm[0, 1]   # good customer predicted bad  -> cost 1 each
print(f"\nFN (lent to bad) = {fn} x {COST_FN} = {fn * COST_FN}")
print(f"FP (refused good) = {fp} x {COST_FP} = {fp * COST_FP}")
print(f"TOTAL COST (logistic regression) = {total_cost(y_test, pred_base)}")

# reference points
print("\nReference costs on the same test set:")
print("  approve everyone:", total_cost(y_test, np.zeros(len(y_test), dtype=int)))
print("  refuse everyone :", total_cost(y_test, np.ones(len(y_test), dtype=int)))


           pred good  pred bad
true good        124        16
true bad          35        25

FN (lent to bad) = 35 x 5 = 175
FP (refused good) = 16 x 1 = 16
TOTAL COST (logistic regression) = 191

Reference costs on the same test set:
  approve everyone: 300
  refuse everyone : 140


**Observation (A2):**

> **Numbers to quote after running:** the confusion matrix, FN and FP counts, total cost, and the approve-everyone / refuse-everyone reference costs.
> Accuracy treats both errors as equal, but lending to a bad customer costs 5 times as much as refusing a good one. Accuracy was therefore the wrong metric: two models with the same accuracy can differ a lot in cost.
> Compare your model's cost with the trivial policies: if "refuse everyone" costs about the same, the 0.5-threshold model is barely paying for itself.


## A3. ROC and AUC  &nbsp;&nbsp;`[2 marks]`

*Dataset: A3.*

a. Plot the ROC curve and report AUC.
b. Mark the operating point that minimises the cost from A2.
c. State in two lines what AUC measures that accuracy does not.


**Observation (A3):**

>

---

# Section B — attempt any 1 of 2 (4 marks each)

*attempt 1 of 2*

## B1. Cost-sensitive training  &nbsp;&nbsp;`[4 marks]`

*Dataset: A3.*

a. Retrain with `class_weight={good:1, bad:5}` and separately by tuning the decision threshold on a validation split.
b. Report accuracy, recall on the bad class and total cost for the baseline and both approaches.
c. State in 4–6 lines which approach you would ship and why.


In [5]:
# Three approaches, all evaluated on the same untouched test set.
rows = []
def report(name, y_pred):
    rows.append({"approach": name,
                 "accuracy": accuracy_score(y_test, y_pred),
                 "recall (bad)": recall_score(y_test, y_pred),
                 "total cost": total_cost(y_test, y_pred)})

# 0. baseline (from A1)
report("baseline (t = 0.5)", pred_base)

# 1. class_weight = {good: 1, bad: 5}
cw = make_pipe(class_weight={0: 1, 1: COST_FN}).fit(X_train, y_train)
report("class_weight {1, 5}", cw.predict(X_test))

# 2. threshold tuned on a VALIDATION split carved out of train (test never used for tuning)
X_fit, X_val, y_fit, y_val = train_test_split(
    X_train, y_train, test_size=0.25, stratify=y_train, random_state=RANDOM_STATE)
thr_model = make_pipe().fit(X_fit, y_fit)
p_val = thr_model.predict_proba(X_val)[:, 1]
grid = np.linspace(0.05, 0.95, 91)
val_costs = [total_cost(y_val, (p_val >= t).astype(int)) for t in grid]
t_best = float(grid[int(np.argmin(val_costs))])
print(f"Threshold chosen on validation: {t_best:.2f}")
p_test = thr_model.predict_proba(X_test)[:, 1]
report(f"tuned threshold (t = {t_best:.2f})", (p_test >= t_best).astype(int))

b1 = pd.DataFrame(rows).set_index("approach")
print(b1.round(4).to_string())


Threshold chosen on validation: 0.22
                            accuracy  recall (bad)  total cost
approach                                                      
baseline (t = 0.5)             0.745        0.4167         191
class_weight {1, 5}            0.650        0.8333         110
tuned threshold (t = 0.22)     0.700        0.8000         108


**Observation (B1):**

> **Numbers to quote after running:** the B1 table (accuracy, recall on bad, total cost for the three approaches) and the validation threshold.
> Both cost-aware approaches should raise recall on the bad class and cut total cost sharply, at the price of lower accuracy and more good customers refused. That drop in accuracy is expected and acceptable under a 5:1 cost.
> **What I would ship:** the threshold-tuned model if its cost is equal or lower, because the threshold can be re-tuned when the business cost ratio changes without retraining, and it is chosen on held-out validation data. class_weight is simpler and needs no extra split, but it changes the fitted coefficients and probabilities (they stop being calibrated).
> State which you picked from your own table, and note the test set is small (200 rows), so differences of a few cost units are within noise.


## B2. Group outcomes  &nbsp;&nbsp;`[4 marks]`

*Dataset: A3.*

a. Split the test set by a protected-style attribute available in the data (for example `personal_status` or age band).
b. Report approval rate, recall and false-positive rate per group in one table.
c. In 6–8 lines, state whether the differences you see are evidence of a problem, and what further information you would need to decide.


**Observation (B2):**




---

# Section C — compulsory (2 marks)

*compulsory*

## C1. Explain one decision  &nbsp;&nbsp;`[2 marks]`

*Dataset: A3.*

a. Pick one applicant your model rejected and list the three features that pushed the score most.
b. Write the two-sentence explanation you would give that applicant.


In [11]:
# a. one rejected applicant and the three features pushing the score hardest toward "bad"

pred_base = base.predict(X_test)
proba_base = base.predict_proba(X_test)[:, 1]

# Find a rejected applicant (predicted class 1)
rejected = np.where(pred_base == 1)[0]

if len(rejected) == 0:
    print("No applicants were rejected.")
else:
    idx = rejected[0]
    applicant = X_test.iloc[[idx]]

    print(
        f"Applicant (test row {X_test.index[idx]}): "
        f"P(bad) = {proba_base[idx]:.3f}, "
        f"true label = {'bad' if y_test.iloc[idx] else 'good'}"
    )

    pre = base.named_steps["pre"]
    clf = base.named_steps["clf"]

    x_transformed = pre.transform(applicant)
    x_values = (
        x_transformed.toarray().ravel()
        if hasattr(x_transformed, "toarray")
        else np.asarray(x_transformed).ravel()
    )
    feature_names = pre.get_feature_names_out()

    contributions = pd.Series(
        clf.coef_[0] * x_values,
        index=feature_names
    )

    print("\nTop 3 features pushing toward REJECT (contribution to log-odds of bad):")
    print(contributions.sort_values(ascending=False).head(3).to_string())

    print("\nApplicant's raw values:")
    print(applicant.iloc[0].to_string())

    print("\nTop 3 features pushing toward APPROVE:")
    print(contributions.sort_values().head(3).to_string())

Applicant (test row 355): P(bad) = 0.740, true label = bad

Top 3 features pushing toward REJECT (contribution to log-odds of bad):
cat__savings_status_<100    0.611974
cat__purpose_new car        0.586105
cat__foreign_worker_yes     0.524181

Applicant's raw values:
checking_status                     0<=X<200
duration                                  24
credit_history                 existing paid
purpose                              new car
credit_amount                           1246
savings_status                          <100
employment                                <1
installment_commitment                     4
personal_status                  male single
other_parties                           none
residence_since                            2
property_magnitude               real estate
age                                       23
other_payment_plans                   stores
housing                                  own
existing_credits                           1
job         

**Observation (C1):**

> **Numbers to quote after running:** the chosen applicant's P(bad) and the top three features from the output.
> **Explanation template (fill in the real features):** "Your application was declined mainly because of [feature 1], [feature 2] and [feature 3], which together raised the estimated risk of default above our threshold. Improving [actionable factor, e.g. shortening the loan duration or building savings] could change the outcome, and you may ask for a manual review."
> Because the model is linear, each feature's contribution is simply coefficient times the encoded value, which makes this explanation faithful. Avoid citing protected attributes such as age as a reason.


---

## Self-check before submitting

- [ ] Kernel restarted and run top-to-bottom, all outputs visible
- [ ] Every attempted question has a written observation
- [ ] No transformer fitted outside a pipeline
- [ ] Metrics reported with units where they have any
- [ ] File named `AML_LA07_<SAPID>.ipynb`
